# Test MCP Servers via MaaS Gateway

This notebook tests MCP server access through the MaaS gateway:
1. MCP endpoint discovery via gateway
2. Authentication enforcement on MCP routes
3. Streamable HTTP connectivity per server
4. Tool invocation tests (Code Sandbox, Codebase Search)
5. Direct Route vs MaaS Gateway comparison

**Prerequisites:**
- MaaS enabled with MCP servers registered (`2_enable_maas.ipynb` Step 6 completed)
- MCP servers deployed in `mcp-servers` namespace (Phase 1)

In [1]:
import subprocess, json, os
from dotenv import load_dotenv

load_dotenv("../.env")

CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN")
if not CLUSTER_DOMAIN:
    r = subprocess.run(["oc", "get", "ingresses.config.openshift.io", "cluster",
                        "-o", "jsonpath={.spec.domain}"], capture_output=True, text=True)
    CLUSTER_DOMAIN = r.stdout.strip()

MAAS_HOST = f"https://maas-api.{CLUSTER_DOMAIN}"

API_KEY = os.getenv("MAAS_API_KEY", "")
if API_KEY and len(API_KEY) > 16:
    api_key_masked = API_KEY[:12] + "..." + API_KEY[-4:]
    print(f"Using MaaS API key: {api_key_masked}")
elif API_KEY:
    print("Using MaaS API key (short)")
else:
    print("⚠️  MAAS_API_KEY not set — run 2_enable_maas.ipynb first")

print(f"MaaS Gateway: {MAAS_HOST}")

Using MaaS API key: sk-oai-1Bq6X...bk0h
MaaS Gateway: https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com


## 1. Discover MCP Endpoints via Gateway

List all MCP servers registered with the MaaS gateway via HTTPRoute.

In [2]:
%%bash
source ../.env 2>/dev/null || true
CLUSTER_DOMAIN=${CLUSTER_DOMAIN:-$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')}
MCP_NS="mcp-servers"
HOST="https://maas-api.${CLUSTER_DOMAIN}"

echo "MCP Servers via MaaS Gateway"
echo "============================================================"
echo ""
printf "%-22s %-10s %s\n" "SERVER" "AIR-GAP" "GATEWAY ENDPOINT"
printf "%-22s %-10s %s\n" "------" "-------" "----------------"

for route in $(oc get httproute -n ${MCP_NS} -l maas.opendatahub.io/managed=true -o jsonpath='{range .items[*]}{.metadata.name}{"\n"}{end}' 2>/dev/null); do
    SHORT_NAME=$(echo $route | sed 's/^mcp-route-//')
    URL="${HOST}/mcp/${SHORT_NAME}/mcp"
    case $SHORT_NAME in
        context7|searxng) AIRGAP="No" ;;
        *) AIRGAP="Yes" ;;
    esac
    printf "%-22s %-10s %s\n" "${SHORT_NAME}" "${AIRGAP}" "${URL}"
done

if [ -z "$(oc get httproute -n ${MCP_NS} -l maas.opendatahub.io/managed=true -o jsonpath='{.items[*].metadata.name}' 2>/dev/null)" ]; then
    echo "No MCP HTTPRoutes registered. Run 2_enable_maas.ipynb Step 6 first."
fi

MCP Servers via MaaS Gateway

SERVER                 AIR-GAP    GATEWAY ENDPOINT
------                 -------    ----------------
code-sandbox           Yes        https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/code-sandbox/mcp
codebase-search        Yes        https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/codebase-search/mcp
context7               No         https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/context7/mcp
searxng             No         https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/searxng/mcp
ocp-mcp-server         Yes        https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/ocp-mcp-server/mcp
repo-docs              Yes        https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/repo-docs/mcp


## 2. Authentication Enforcement

Verify that MCP endpoints via the gateway require a valid credential. Both **MaaS API keys** and **OCP tokens** are accepted.

In [3]:
%%bash
source ../.env 2>/dev/null || true
CLUSTER_DOMAIN=${CLUSTER_DOMAIN:-$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')}
HOST="https://maas-api.${CLUSTER_DOMAIN}"
MCP_NS="mcp-servers"

FIRST_ROUTE=$(oc get httproute -n ${MCP_NS} -l maas.opendatahub.io/managed=true -o jsonpath='{.items[0].metadata.name}' 2>/dev/null)
SHORT_NAME=$(echo $FIRST_ROUTE | sed 's/^mcp-route-//')
URL="${HOST}/mcp/${SHORT_NAME}/mcp"

echo "Testing auth enforcement on: ${URL}"
echo ""

echo "1. No auth header (expecting 401/403):"
HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 "${URL}")
if [ "$HTTP_CODE" = "401" ] || [ "$HTTP_CODE" = "403" ]; then
    echo "   PASS — Rejected (HTTP ${HTTP_CODE})"
else
    echo "   Got HTTP ${HTTP_CODE} — auth may not be enforced on MCP routes"
fi

echo ""
echo "2. Invalid API key (expecting 401/403):"
HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 \
  -H "Authorization: Bearer sk-oai-INVALID" "${URL}")
if [ "$HTTP_CODE" = "401" ] || [ "$HTTP_CODE" = "403" ]; then
    echo "   PASS — Invalid key rejected (HTTP ${HTTP_CODE})"
else
    echo "   Got HTTP ${HTTP_CODE}"
fi

INIT_BODY='{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-03-26","capabilities":{},"clientInfo":{"name":"auth-test","version":"1.0"}}}'

echo ""
echo "3. Valid MaaS API key — MCP initialize (expecting 200):"
if [ -n "${MAAS_API_KEY}" ]; then
    HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 10 -X POST \
      -H "Authorization: Bearer ${MAAS_API_KEY}" \
      -H "Content-Type: application/json" \
      -H "Accept: application/json, text/event-stream" \
      -d "${INIT_BODY}" "${URL}")
    if [ "$HTTP_CODE" = "200" ]; then
        echo "   PASS — API key accepted, MCP initialize OK (HTTP 200)"
    elif [ "$HTTP_CODE" = "429" ]; then
        echo "   PASS — API key accepted, rate limited (HTTP 429)"
    else
        echo "   Got HTTP ${HTTP_CODE}"
    fi
else
    echo "   SKIP — MAAS_API_KEY not set in .env"
fi

echo ""
echo "4. Valid OCP token — MCP initialize (expecting 200):"
HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 10 -X POST \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -d "${INIT_BODY}" "${URL}")
if [ "$HTTP_CODE" = "200" ]; then
    echo "   PASS — OCP token accepted, MCP initialize OK (HTTP 200)"
elif [ "$HTTP_CODE" = "429" ]; then
    echo "   PASS — OCP token accepted, rate limited (HTTP 429)"
else
    echo "   Got HTTP ${HTTP_CODE}"
fi

Testing auth enforcement on: https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/code-sandbox/mcp

1. No auth header (expecting 401/403):
   PASS — Rejected (HTTP 401)

2. Invalid API key (expecting 401/403):
   PASS — Invalid key rejected (HTTP 403)

3. Valid MaaS API key — MCP initialize (expecting 200):
   PASS — API key accepted, MCP initialize OK (HTTP 200)

4. Valid OCP token — MCP initialize (expecting 200):
   PASS — OCP token accepted, MCP initialize OK (HTTP 200)


## 3. Test MCP Protocol — Code Sandbox

Send MCP `initialize` + `tools/call` to Code Sandbox via the gateway.

In [4]:
%%bash
source ../.env 2>/dev/null || true
CLUSTER_DOMAIN=${CLUSTER_DOMAIN:-$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')}
HOST="https://maas-api.${CLUSTER_DOMAIN}"
URL="${HOST}/mcp/code-sandbox/mcp"
TOKEN="${MAAS_API_KEY:-$(oc whoami -t)}"

echo "Testing Code Sandbox via MaaS gateway: ${URL}"
echo ""

# Initialize
echo "1. Initialize:"
INIT_RESP=$(curl -sSk -m 10 -X POST \
  -H "Authorization: Bearer ${TOKEN}" \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -d '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-03-26","capabilities":{},"clientInfo":{"name":"test","version":"1.0"}}}' \
  "${URL}")

if echo "$INIT_RESP" | grep -q 'serverInfo\|protocolVersion'; then
    echo "   PASS — MCP initialize successful"
else
    echo "   Response: ${INIT_RESP:0:200}"
fi

echo ""

# Execute code
echo "2. Execute Python code:"
EXEC_RESP=$(curl -sSk -m 10 -X POST \
  -H "Authorization: Bearer ${TOKEN}" \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -d '{"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"execute_code","arguments":{"code":"print(2+2)","language":"python"}}}' \
  "${URL}")

if echo "$EXEC_RESP" | grep -q '4\|result'; then
    echo "   PASS — Code execution returned result"
fi
echo "   Response (first 300 chars):"
echo "   ${EXEC_RESP:0:300}"

Testing Code Sandbox via MaaS gateway: https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/code-sandbox/mcp

1. Initialize:
   PASS — MCP initialize successful

2. Execute Python code:
   PASS — Code execution returned result
   Response (first 300 chars):
   event: message
data: {"jsonrpc":"2.0","id":2,"result":{"content":[{"type":"text","text":"[python] OK (0.01s)\n\nstdout:\n4"}],"isError":false}}



## 4. Test MCP Protocol — Codebase Search

Test semantic code search over the indexed cafe-order-system.

In [5]:
%%bash
source ../.env 2>/dev/null || true
CLUSTER_DOMAIN=${CLUSTER_DOMAIN:-$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')}
HOST="https://maas-api.${CLUSTER_DOMAIN}"
URL="${HOST}/mcp/codebase-search/mcp"
TOKEN="${MAAS_API_KEY:-$(oc whoami -t)}"

echo "Testing Codebase Search via MaaS gateway: ${URL}"
echo ""

# Initialize
curl -sSk -m 10 -X POST \
  -H "Authorization: Bearer ${TOKEN}" \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -d '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-03-26","capabilities":{},"clientInfo":{"name":"test","version":"1.0"}}}' \
  "${URL}" > /dev/null 2>&1

# Search code
echo "Query: 'order creation logic'"
SEARCH_RESP=$(curl -sSk -m 15 -X POST \
  -H "Authorization: Bearer ${TOKEN}" \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -d '{"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"search_code","arguments":{"query":"order creation logic","top_k":2}}}' \
  "${URL}")

if echo "$SEARCH_RESP" | grep -q 'result\|content'; then
    echo "PASS — Semantic search returned results"
    echo ""
    echo "Response (first 500 chars):"
    echo "${SEARCH_RESP:0:500}"
else
    echo "No results or error:"
    echo "${SEARCH_RESP:0:300}"
fi

Testing Codebase Search via MaaS gateway: https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/codebase-search/mcp

Query: 'order creation logic'
PASS — Semantic search returned results

Response (first 500 chars):
event: message
data: {"jsonrpc":"2.0","id":2,"result":{"content":[{"type":"text","text":"--- Result 1 (score: 0.378) ---\nFile: ..2026_06_18_08_45_36.1191552457/routes_orders.py (lines 36-75)\n\n@router.post(\"/\", response_model=OrderResponse, status_code=201)\ndef create_order(order_data: OrderCreate, db: Session = Depends(get_db)):\n    if len(order_data.items) > MAX_ITEMS_PER_ORDER:\n        raise HTTPException(\n            status_code=400,\n            detail=f\"Maximum {MAX_ITEMS_PER_ORD


## 5. Direct Route vs MaaS Gateway Comparison

Compare direct access (no auth) with gateway access (auth required).

In [6]:
%%bash
source ../.env 2>/dev/null || true
CLUSTER_DOMAIN=${CLUSTER_DOMAIN:-$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')}
MCP_NS="mcp-servers"
MAAS_HOST="https://maas-api.${CLUSTER_DOMAIN}"

TOKEN="${MAAS_API_KEY:-}"
INIT_BODY='{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-03-26","capabilities":{},"clientInfo":{"name":"test","version":"1.0"}}}'

echo "Direct Route vs MaaS Gateway (MCP initialize POST)"
echo "============================================================"
echo ""
printf "%-22s %-14s %-14s %-14s\n" "SERVER" "DIRECT(open)" "GW(no auth)" "GW(API key)"
printf "%-22s %-14s %-14s %-14s\n" "------" "------------" "-----------" "-----------"

for route in $(oc get routes -n ${MCP_NS} -o jsonpath='{range .items[*]}{.metadata.name}{"\n"}{end}' 2>/dev/null); do
    host=$(oc get route $route -n ${MCP_NS} -o jsonpath='{.spec.host}')
    DIRECT_URL="https://${host}/mcp"
    SHORT_NAME=$(echo $route | sed 's/^mcp-//')
    MAAS_URL="${MAAS_HOST}/mcp/${SHORT_NAME}/mcp"

    DIRECT_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 10 -X POST \
      -H "Content-Type: application/json" -H "Accept: application/json, text/event-stream" \
      -d "${INIT_BODY}" "${DIRECT_URL}")
    NOAUTH_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 "${MAAS_URL}")

    if [ -n "$TOKEN" ]; then
        AUTH_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 10 -X POST \
          -H "Authorization: Bearer ${TOKEN}" \
          -H "Content-Type: application/json" -H "Accept: application/json, text/event-stream" \
          -d "${INIT_BODY}" "${MAAS_URL}")
    else
        AUTH_CODE="SKIP"
    fi

    printf "%-22s HTTP %-8s HTTP %-8s HTTP %-8s\n" "${route}" "${DIRECT_CODE}" "${NOAUTH_CODE}" "${AUTH_CODE}"
done

echo ""
echo "Expected:"
echo "  DIRECT(open):  200 — MCP initialize succeeds (no auth required)"
echo "  GW(no auth):   401 — rejected without credentials"
echo "  GW(API key):   200 — MCP initialize succeeds (API key authenticated)"
echo ""
echo "Same API key works for both model inference and MCP tool access."

Direct Route vs MaaS Gateway (MCP initialize POST)

SERVER                 DIRECT(open)   GW(no auth)    GW(API key)   
------                 ------------   -----------    -----------   
mcp-code-sandbox       HTTP 200      HTTP 401      HTTP 200     
mcp-codebase-search    HTTP 200      HTTP 401      HTTP 200     
mcp-context7           HTTP 200      HTTP 401      HTTP 200     
mcp-searxng         HTTP 200      HTTP 401      HTTP 200     
mcp-repo-docs          HTTP 200      HTTP 401      HTTP 200     
ocp-mcp-server         HTTP 200      HTTP 401      HTTP 200     

Expected:
  DIRECT(open):  200 — MCP initialize succeeds (no auth required)
  GW(no auth):   401 — rejected without credentials
  GW(API key):   200 — MCP initialize succeeds (API key authenticated)

Same API key works for both model inference and MCP tool access.


## Summary

| Test | What It Validates |
|------|-------------------|
| Endpoint Discovery | MCP HTTPRoutes registered with gateway |
| Auth Enforcement | MCP routes reject unauthenticated requests |
| Code Sandbox | Streamable HTTP + code execution via gateway |
| Codebase Search | Semantic search tool call via gateway |
| Direct vs Gateway | Auth difference between open routes and MaaS |

## Next Steps

- `../4_control/1_maas_advanced.ipynb` — Subscription management and multi-tier demo
- `../4_control/2_maas_policy_test.ipynb` — Rate limit enforcement testing